# Multi-Representation Search: Step-by-Step Build-Up

A document is rarely well-represented by a single embedding. A research paper has a title, an abstract, body chunks, and category tags. Each carries a different signal, and squashing all four into one dense vector loses most of that structure: the title gets averaged out, keyword matches on tags disappear, and chunk-level grounding for downstream reasoning is gone.

This notebook builds a Qdrant retrieval pipeline that uses each representation deliberately. Over five steps you'll go from a naive dense-only baseline to a fully fused pipeline with three named-vector prefetches, Reciprocal Rank Fusion, document-level grouping, and optional formula-based score boosting. After each step you'll run the same query and see the top retrieved papers change.

The design rationale (why each component is there, when to use it, when not to) lives in the accompanying [tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/multi-representation-search/). This notebook focuses on running the code and watching the result list shift.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/qdrant/examples/blob/master/multi-representation-search/multi-representation-search.ipynb)


## Requirements

Use Python <3.13. Not all dependencies support the newest Python versions yet.


In [ ]:
!pip install qdrant-client fastembed datasets

## Dataset

20 000 arXiv papers from the [`gfissore/arxiv-abstracts-2021`](https://huggingface.co/datasets/gfissore/arxiv-abstracts-2021) Hugging Face dataset, filtered to ML/CS and to papers from 2018 onward. Each paper exposes a `title`, `abstract`, and `categories` (which this dataset returns as space-joined strings, so we split them before filtering). Swap in any other arXiv source as long as it exposes those three fields.

In [ ]:
from datasets import load_dataset

ML_CATEGORIES = {"cs.LG", "cs.CV", "cs.CL", "cs.AI", "stat.ML"}

# Non-streaming so HF caches the parquet locally; first run downloads ~2.5 GB, re-runs are instant.
dataset = load_dataset("gfissore/arxiv-abstracts-2021", split="train")

papers = []
# IDs are roughly chronological; iterate from the end to land on 2021/2020/2019 papers first.
for i in range(len(dataset) - 1, -1, -1):
    if len(papers) >= 20000:
        break
    row = dataset[i]
    if not row["abstract"] or not row["title"]:
        continue
    # categories arrive as space-joined strings (e.g. ["cs.LG cs.CV"]); split each entry.
    cats = [tok for entry in row["categories"] for tok in entry.split()]
    if not any(c in ML_CATEGORIES for c in cats):
        continue
    # Year lives in the YYMM prefix of new-format arXiv IDs ("2104.01234" -> 2021).
    arxiv_id = row["id"]
    if "/" in arxiv_id or "." not in arxiv_id:
        continue  # skip pre-2007 IDs like "math/0506001"
    if 2000 + int(arxiv_id[:2]) < 2018:
        continue
    papers.append({
        "arxiv_id": arxiv_id,
        "title": row["title"].strip(),
        "abstract": row["abstract"].strip(),
        "categories": cats,
    })
print(f"Loaded {len(papers)} papers")


## Schema

One Qdrant collection. Each point is a chunk. Each chunk holds four named vectors that we'll fuse at query time:

- `dense_chunk`: the chunk's own embedding (body content).
- `dense_title`: the paper title embedding (topical naming).
- `dense_summary`: the paper abstract embedding (contribution focus).
- `sparse_keywords`: BM25 over the title and tags concatenated (lexical matches on short structured fields).

`dense_title` and `dense_summary` are duplicated across every chunk of the same paper. That trades a bit of storage for one-shot query fusion (one collection, one Query API call, no `lookup_from`). For the typical case (a few dozen chunks per paper, embeddings under a kilobyte each) it's the simpler choice.

We use *named vectors*, not a multivector field. Multivectors are designed for late-interaction models like ColBERT, where the MaxSim comparator combines per-token subvectors into one score per point. Title, summary, and chunk vectors are different kinds of content, so MaxSim would collapse the per-representation signal we want to fuse. The [tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/multi-representation-search/) covers the contrast.


In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient("http://localhost:6333")  # or QdrantClient(url="https://<id>.cloud.qdrant.io", api_key="...") for Qdrant Cloud

client.create_collection(
    collection_name="arxiv_multi_repr",
    vectors_config={
        "dense_chunk":   models.VectorParams(size=384, distance=models.Distance.COSINE),
        "dense_title":   models.VectorParams(size=384, distance=models.Distance.COSINE),
        "dense_summary": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse_keywords": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

## Ingestion

Embeddings are generated locally with [FastEmbed](https://qdrant.tech/documentation/fastembed/):

- `BAAI/bge-small-en-v1.5` (384-dim, ~67 MB) for the three dense vectors. Trained with retrieval-specific contrastive objectives, which is what this tutorial does.
- `Qdrant/bm25` for the sparse vector. The IDF modifier on the collection means Qdrant computes inverse-document-frequency weights at query time across the corpus.

Chunking uses a fixed two-sentence window for clarity. Chunking strategy has a real effect on retrieval quality and is its own design space (hierarchical, late, semantic chunking are all worth comparing). For now: one point per chunk, with the title and summary embeddings copied onto every chunk of the same paper.

The loop is deliberately straightforward (one paper at a time) so the per-vector logic stays easy to follow. The first run downloads the FastEmbed models; subsequent runs reuse the local cache. On a laptop CPU expect roughly 15–20 minutes for 20000 papers.

In [ ]:
from fastembed import TextEmbedding, SparseTextEmbedding

# Dense embeddings for title, summary, and chunk content; sparse BM25 for keyword matching.
dense_model = TextEmbedding("BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding("Qdrant/bm25")

def chunk_sentences(text, target_len=2):
    """Split text into ~2-sentence chunks; fall back to the full text if it doesn't split cleanly."""
    sentences = [s.strip() for s in text.split(". ") if s.strip()]
    return [". ".join(sentences[i:i + target_len])
            for i in range(0, len(sentences), target_len)] or [text]

def to_sparse(sparse_emb):
    """Convert FastEmbed's SparseEmbedding into a Qdrant SparseVector."""
    return models.SparseVector(
        indices=sparse_emb.indices.tolist(),
        values=sparse_emb.values.tolist(),
    )


points = []
for paper in papers:
    chunks = chunk_sentences(paper["abstract"])

    # Paper-level embeddings: computed once per paper, reused across every chunk below.
    # next(iter(...)) extracts the single vector from FastEmbed's generator output.
    title_vec   = next(iter(dense_model.embed([paper["title"]]))).tolist()
    summary_vec = next(iter(dense_model.embed([paper["abstract"]]))).tolist()
    sparse_vec  = to_sparse(next(iter(sparse_model.embed(
        [paper["title"] + " " + " ".join(paper["categories"])]
    ))))

    # Chunk-level dense embedding: one vector per chunk.
    chunk_vecs = [v.tolist() for v in dense_model.embed(chunks)]

    # One Qdrant point per chunk. dense_title, dense_summary, and sparse_keywords
    # are the same for every chunk of this paper; only dense_chunk varies.
    for i, (chunk, chunk_vec) in enumerate(zip(chunks, chunk_vecs)):
        points.append(models.PointStruct(
            id=len(points),
            vector={
                "dense_chunk":     chunk_vec,
                "dense_title":     title_vec,
                "dense_summary":   summary_vec,
                "sparse_keywords": sparse_vec,
            },
            payload={
                "document_id": paper["arxiv_id"],
                "title":       paper["title"],
                "tags":        paper["categories"],
                "chunk_index": i,
                "chunk_text":  chunk,
            },
        ))

client.upload_points(collection_name="arxiv_multi_repr", points=points, batch_size=64)
print(f"Uploaded {len(points)} chunks across {len(papers)} papers")


## Query Helpers

Three pieces used by every step below:

- `embed_query(query)` produces the `(dense, sparse)` pair we feed into Qdrant. Both `dense_model` and `sparse_model` expose a `query_embed` method calibrated for queries: for BM25 it applies IDF weighting; for some dense models it applies a query-side prompt.
- `SAMPLE_QUERY` is the single query we run through every step so we can watch the same query produce different results as capabilities are added.
- `show_results(retrieve_fn)` runs the retrieve function and prints the top 5 results: title, category tags, and an excerpt from the matching chunk. Accepts both chunk-level results (Steps 1-3) and grouped results (Steps 4-5, where each result is a paper with several chunks).


In [ ]:
import textwrap

def embed_query(query):
    """Produce a (dense, sparse) embedding pair for a query string."""
    dense = next(iter(dense_model.query_embed([query]))).tolist()
    sparse = to_sparse(next(iter(sparse_model.query_embed([query]))))
    return dense, sparse

SAMPLE_QUERY = "diffusion models for image synthesis"

def show_results(retrieve_fn, query=SAMPLE_QUERY, k=5):
    """Print top-k results as: title, category tags, and a matching-chunk excerpt."""
    print(f"Query: {query!r}\n")
    for i, item in enumerate(retrieve_fn(query, limit=k), 1):
        # item is a Point (Steps 1-3) or a Group (Steps 4-5).
        # For groups, hits[0] is the top chunk for that paper.
        point = item.hits[0] if hasattr(item, "hits") else item
        payload = point.payload
        title = payload["title"]
        tags = payload.get("tags", [])
        # Collapse whitespace (including embedded newlines) so the excerpt prints cleanly.
        chunk = " ".join(payload["chunk_text"].split())
        excerpt = chunk[:250].rstrip() + ("..." if len(chunk) > 250 else "")
        print(textwrap.fill(f"{i}. {title}", width=140, initial_indent="  ", subsequent_indent="     "))
        if tags:
            print(f"     [{', '.join(str(t) for t in tags[:3])}]")
        print(textwrap.fill(excerpt, width=140, initial_indent="     ", subsequent_indent="     "))
        print()


## Step 1: Dense Over Chunks (Baseline)

The naive baseline: encode the query with the dense model, search against `dense_chunk` only, return the chunk-level results' parent papers. No fusion, no title or sparse signal.

This is what most "vector search" tutorials stop at. It's a reasonable default for short, homogeneous corpora where the chunk text already carries the full signal. It systematically underperforms when the signal lives outside the chunk: in the title (topical naming), in tags (controlled vocabulary), or in keyword overlap that the embedding model has averaged out into a generic neighborhood.

Each subsequent step closes one of those gaps.


In [ ]:
def retrieve_baseline(query, limit=10):
    dense, _ = embed_query(query)
    return client.query_points(
        collection_name="arxiv_multi_repr",
        query=dense,
        using="dense_chunk",
        limit=limit,
    ).points

show_results(retrieve_baseline)


## Step 2: Add Sparse Keywords With RRF

Add a second prefetch: BM25 over title and tags. Then fuse the two ranked lists with **Reciprocal Rank Fusion (RRF)**.

Why RRF instead of weighted averages of raw scores? RRF works on rank, not score. Dense scores live in [0, 1], sparse BM25 scores don't, and RRF doesn't have to reconcile the two. Linear weights are fragile: a weight that helps one query class hurts another, and the right weight depends on query length, model, and corpus.

What does sparse add? Queries with rare entity names, jargon, or category tags often produce dense embeddings near generic neighborhoods. The sparse path catches those exact-token matches. RRF promotes documents both paths agree on.


In [ ]:
def retrieve_hybrid(query, limit=10):
    dense, sparse = embed_query(query)
    return client.query_points(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense,  using="dense_chunk",     limit=50),
            models.Prefetch(query=sparse, using="sparse_keywords", limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
    ).points

show_results(retrieve_hybrid)


## Step 3: Add Title Prefetch

Add a third prefetch: the same dense query vector, but searched against `dense_title` instead of `dense_chunk`. We're now fusing across three representations: chunk content, keyword hits, and topical naming.

The title prefetch saves queries where the topic is named explicitly but not echoed in any single chunk. For example: "diffusion models for high-resolution image synthesis" surfaces a paper titled "High-Resolution Image Synthesis with Latent Diffusion Models" via the title path even when its chunks phrase the contribution differently. The chunk prefetch alone misses it; the title path catches it; RRF promotes it because both paths agree.

A representation only earns its own prefetch if it carries signal independent of the others. We're not adding `dense_summary` as a fourth prefetch here because abstracts often paraphrase the chunks they came from. If your corpus has summaries that surface different content (human-written summaries of long technical reports, for example), adding a fourth prefetch is worth it.


In [ ]:
def retrieve_three_repr(query, limit=10):
    dense, sparse = embed_query(query)
    return client.query_points(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense,  using="dense_chunk",     limit=50),
            models.Prefetch(query=dense,  using="dense_title",     limit=50),
            models.Prefetch(query=sparse, using="sparse_keywords", limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
    ).points

show_results(retrieve_three_repr)


## Step 4: Group by Document

So far results are chunks, and the same paper can appear multiple times in the top 10. Most consumers want one entry per document with the top chunks attached: a results UI, a citation list, an LLM that needs document-level attribution.

`query_points_groups` collapses chunks back to documents using `group_by="document_id"`. Each group's `hits` field carries the top-`group_size` chunks for that paper.

A few things worth knowing:

- Grouping is a *presentation* choice, not a relevance technique. The candidates and their fused scores don't change; only the result shape does.
- Increase the prefetch `limit` when grouping. If a paper has three good chunks but the prefetch only returned two, grouping doesn't have the third to consider.
- Use the `with_lookup` parameter when document-level metadata (full title, authors, dates) lives in a separate collection. It fetches one record per group instead of repeating it per chunk.

When *not* to group: when an LLM benefits from seeing several independently ranked chunks across multiple documents in its context window. Collapsing those into per-document groups throws away ordering information the LLM could have used.


In [ ]:
def retrieve_grouped(query, limit=10, group_size=3):
    dense, sparse = embed_query(query)
    return client.query_points_groups(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense,  using="dense_chunk",     limit=100),
            models.Prefetch(query=dense,  using="dense_title",     limit=100),
            models.Prefetch(query=sparse, using="sparse_keywords", limit=100),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        group_by="document_id",
        group_size=group_size,
        limit=limit,
    ).groups

show_results(retrieve_grouped)


## Step 5: Score Boosting With a Formula

When you have ranking preferences that aren't captured by similarity alone (recency, source authority, geographic proximity, structured boosts), swap RRF for a `FormulaQuery`. Formulas operate on the prefetch scores and payload fields:

- `$score[i]` references the score from prefetch `i`. Prefetch order is load-bearing.
- The `defaults` map covers candidates that appeared in one prefetch but not another. Without it, a missing variable would error.

The formula below sums the chunk score with a half-weighted title score and a smaller sparse contribution. Unlike RRF, this is a linear combination of raw scores and is fragile across query types unless you've held the weights up against representative queries. Treat the specific weights here as illustrative; the mechanism is the point.

Formula vs reranker:

- **Formula API**: structured preferences known up front (recency decay, source authority, geo proximity, content-type boosts). Cheap and deterministic.
- **Reranker** (a late-interaction or cross-encoder model): preferences that are "this is more relevant than that" but you can't easily express why in a closed form. Expensive but learns what you can't articulate.

For time decay on a `published_at` payload field, swap the title term for an `exp_decay` expression from Qdrant's [decay functions reference](https://qdrant.tech/documentation/search/search-relevance/#decay-functions).


In [ ]:
def retrieve_boosted(query, limit=10, group_size=3):
    dense, sparse = embed_query(query)
    return client.query_points_groups(
        collection_name="arxiv_multi_repr",
        prefetch=[
            # $score[0] = chunk, $score[1] = title, $score[2] = sparse
            models.Prefetch(query=dense,  using="dense_chunk",     limit=100),
            models.Prefetch(query=dense,  using="dense_title",     limit=100),
            models.Prefetch(query=sparse, using="sparse_keywords", limit=100),
        ],
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=[
                "$score[0]",
                models.MultExpression(mult=[0.5, "$score[1]"]),
                models.MultExpression(mult=[0.3, "$score[2]"]),
            ]),
            defaults={"$score[1]": 0.0, "$score[2]": 0.0},
        ),
        group_by="document_id",
        group_size=group_size,
        limit=limit,
    ).groups

show_results(retrieve_boosted)


## Wrap-up

That's the recommended multi-representation pipeline end to end. The same schema works for any corpus with title-like, summary-like, and body-like representations. Swap the dataset, retune which representations earn their prefetch slots for your data, and wire in formula-based ranking preferences as needed.

For the design rationale and references, see the [tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/multi-representation-search/).
